# TRNA Sentiment Indicators

Fetches Thomson Reuters News Analytics (TRNA) per-article fields for a given ticker and
computes the **Sentiment Shock** and **Sentiment Trend** indicators used by Song, Liu & Yang (2017).

**Paper sentiment score** (Eq. 8):
```
S_sentiment = relevance × (pos - neg)
```

**Shock indicator** (Eq. 1, weekly):
```
S_shock(t) = (S(t) - μ(t-N, t-1)) / σ(t-N, t-1)
```

**Trend indicator** (Eq. 2, weekly):
```
S_trend(t) = Σ ΔS(i)  for i in [t-N, t-1]
```

The TRNA fields (`pos`, `neg`, `obj`, `relevance`) can come from two sources:
- **Source A**: Structured analytics attached to headlines via `ld.news.get_headlines()` (if the subscription includes TRNA metadata)
- **Source B**: Per-story metadata via `ld.news.get_story()` with `Format.STRUCTURED` (if available)

This notebook first **discovers** which source is available, then fetches the full 2003–2014 window.

In [ ]:
import json
import time
from pathlib import Path

import lseg.data as ld
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sentiment_ltr.data.news_coverage import (
    build_news_query,
    fetch_headlines_for_window,
    month_chunks,
)
from sentiment_ltr.data.refinitiv_queries import ticker_to_ric_candidates
from sentiment_ltr.data.refinitiv_session import open_refinitiv_session

PROJECT_ROOT = Path("__file__").resolve().parent.parent

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
TICKER      = "AAPL"
START       = "2003-01-01"
END         = "2014-12-31"

# Look-back window N (weeks) for shock / trend computation
N_WEEKS = 8

# Discovery smoke-test window (short)
SMOKE_START = "2006-01-01"
SMOKE_END   = "2006-01-31"

# Rate limit: seconds between story analytics calls
STORY_DELAY = 0.2

# Output paths (gitignored data/raw/)
OUT_DIR = PROJECT_ROOT / "data" / "raw" / "news" / "trna"
OUT_DIR.mkdir(parents=True, exist_ok=True)

slug = TICKER.lower()
ARTICLE_PARQUET   = OUT_DIR / f"{slug}_trna_{START[:4]}_{END[:4]}.parquet"
WEEKLY_PARQUET    = OUT_DIR / f"{slug}_weekly_sentiment_{START[:4]}_{END[:4]}.parquet"
FEATURES_PARQUET  = OUT_DIR / f"{slug}_rsi_features_{START[:4]}_{END[:4]}.parquet"
SUMMARY_JSON      = OUT_DIR / f"{slug}_trna_summary.json"

print(f"Ticker : {TICKER}")
print(f"Window : {START} → {END}")
print(f"Output : {OUT_DIR}")

In [ ]:
# ── Open Refinitiv session ─────────────────────────────────────────────────────
open_refinitiv_session(PROJECT_ROOT, ld)
print("Session opened.")

## Step 1 — Discover available TRNA fields

Fetch a small sample of headlines and inspect all columns. TRNA subscriptions surface
`sentimentPositive`, `sentimentNegative`, `sentimentNeutral`, and `relevance` directly on the
headline row. If those columns are absent, fall back to querying structured story metadata.

In [ ]:
# ── Discover: headline-level TRNA metadata ─────────────────────────────────────
ric_candidates = ticker_to_ric_candidates(TICKER)
smoke_ric = None
smoke_raw = None

for ric in ric_candidates:
    query = build_news_query(ric)
    start_ts = pd.Timestamp(SMOKE_START).to_pydatetime()
    end_ts   = (pd.Timestamp(SMOKE_END) + pd.Timedelta(days=1)).to_pydatetime()
    try:
        raw = ld.news.get_headlines(query, start=start_ts, end=end_ts, count=50)
    except Exception as exc:
        print(f"  {ric}: {exc}")
        continue
    if raw is not None and not (isinstance(raw, pd.DataFrame) and raw.empty):
        smoke_raw = raw if isinstance(raw, pd.DataFrame) else getattr(raw, 'data', raw)
        smoke_ric = ric
        break

if smoke_raw is None:
    print("No headlines returned in smoke window — check session / RIC.")
else:
    print(f"RIC selected : {smoke_ric}")
    df_smoke = smoke_raw if isinstance(smoke_raw, pd.DataFrame) else smoke_raw.df
    print(f"Rows         : {len(df_smoke)}")
    print(f"Columns      : {list(df_smoke.columns)}")
    df_smoke.head(3)

In [ ]:
# ── Check which TRNA analytics fields are present ─────────────────────────────
TRNA_HEADLINE_FIELDS = {
    "sentimentPositive": "pos",
    "sentimentNegative": "neg",
    "sentimentNeutral":  "obj",
    "relevance":         "relevance",
    # Alternative column names some Refinitiv versions use
    "SentimentPositive": "pos",
    "SentimentNegative": "neg",
    "SentimentNeutral":  "obj",
    "Relevance":         "relevance",
    "positiveProb":      "pos",
    "negativeProb":      "neg",
    "neutralProb":       "obj",
}

available_trna_cols = {}
if df_smoke is not None:
    for raw_col, canonical in TRNA_HEADLINE_FIELDS.items():
        if raw_col in df_smoke.columns:
            available_trna_cols[canonical] = raw_col

HAVE_HEADLINE_TRNA = bool(available_trna_cols)
print(f"TRNA analytics in headlines: {HAVE_HEADLINE_TRNA}")
if HAVE_HEADLINE_TRNA:
    print(f"  Fields mapped: {available_trna_cols}")
else:
    print("  ⚠  TRNA probabilities not attached to headlines.")
    print("  Will attempt structured story-level analytics next.")

In [ ]:
# ── Discover: structured story-level analytics ─────────────────────────────────
# Try fetching one story with Format.STRUCTURED to see what analytics fields exist.

HAVE_STRUCTURED_ANALYTICS = False
structured_example = None

if not HAVE_HEADLINE_TRNA and df_smoke is not None and "storyId" in df_smoke.columns:
    sample_ids = df_smoke["storyId"].dropna().head(3).tolist()
    for sid in sample_ids:
        try:
            result = ld.news.get_story(str(sid), format=ld.news.Format.TEXT)
            print(f"Story {sid[:30]}...")
            print(f"  type : {type(result)}")
            if hasattr(result, '__dict__'):
                print(f"  attrs: {list(result.__dict__.keys())}")
            elif isinstance(result, dict):
                print(f"  keys : {list(result.keys())}")
            elif isinstance(result, str):
                print(f"  text preview: {result[:120]}")
            structured_example = result
            break
        except Exception as exc:
            print(f"  {sid[:30]}...: {exc}")
            continue

    # Try STRUCTURED format if available
    if sample_ids:
        try:
            fmt = getattr(ld.news.Format, 'STRUCTURED', None)
            if fmt is not None:
                result_struct = ld.news.get_story(str(sample_ids[0]), format=fmt)
                print(f"\nStructured format type : {type(result_struct)}")
                if isinstance(result_struct, dict):
                    print(f"Keys: {list(result_struct.keys())}")
                    HAVE_STRUCTURED_ANALYTICS = True
                    structured_example = result_struct
            else:
                print("Format.STRUCTURED not available in this SDK version.")
        except Exception as exc:
            print(f"Structured format attempt: {exc}")
else:
    print("Skipping structured story check (headline TRNA already present or no stories found).")

In [ ]:
# ── Discover: pre-aggregated TRNA via get_data() ───────────────────────────────
# Some TRNA subscriptions expose daily/weekly aggregated analytics through TR. fields.
# These are the most convenient but are only available with the TRNA time-series add-on.

TRNA_TS_FIELDS = [
    "TR.News.TRNA.Buzz",
    "TR.News.TRNA.SentimentScore",
    "TR.News.TRNA.BuzzScoreAvg",
    "TR.News.TRNA.NoveltyNV",
]

trna_ts_df = None
try:
    trna_ts_df = ld.get_history(
        universe=[smoke_ric or "AAPL.O"],
        fields=TRNA_TS_FIELDS,
        interval="1W",
        start=SMOKE_START,
        end=SMOKE_END,
    )
    print("get_history with TRNA fields succeeded:")
    print(f"  shape  : {trna_ts_df.shape}")
    print(f"  columns: {list(trna_ts_df.columns)}")
    display(trna_ts_df.head())
    HAVE_TRNA_TS = not trna_ts_df.empty and trna_ts_df.notna().any().any()
    print(f"  Data present: {HAVE_TRNA_TS}")
except Exception as exc:
    print(f"get_history TRNA fields: {exc}")
    HAVE_TRNA_TS = False

In [ ]:
# ── Discovery summary ─────────────────────────────────────────────────────────
print("=" * 60)
print("TRNA availability summary")
print("=" * 60)
print(f"  Headline-level TRNA probabilities : {HAVE_HEADLINE_TRNA}")
print(f"  Structured story analytics        : {HAVE_STRUCTURED_ANALYTICS}")
print(f"  Pre-aggregated TRNA time-series   : {HAVE_TRNA_TS}")
print()

if HAVE_TRNA_TS:
    print("→ Using pre-aggregated TRNA time-series (most efficient path).")
    FETCH_MODE = "trna_ts"
elif HAVE_HEADLINE_TRNA:
    print("→ Using per-article TRNA probabilities from headline metadata.")
    FETCH_MODE = "headline_trna"
else:
    print("→ TRNA analytics not available in this subscription.")
    print("  To replicate the paper, you need TRNA access.")
    print("  Options:")
    print("    1. Request TRNA access from U of T / LSEG.")
    print("    2. Use FinBERT / VADER on fetched article text as a proxy.")
    print("       See notebooks/fetch_news_articles.ipynb for article text pipeline.")
    FETCH_MODE = "unavailable"

print(f"\nFETCH_MODE = {FETCH_MODE!r}")

## Step 2 — Fetch full 2003–2014 TRNA data

This section runs whichever fetch mode was detected above.

In [ ]:
# ── Path A: pre-aggregated TRNA time-series ───────────────────────────────────
# Available when subscription includes TRNA analytics (TR.News.TRNA.*)

weekly_sentiment = None

if FETCH_MODE == "trna_ts":
    # These fields provide weekly Buzz and SentimentScore directly.
    # SentimentScore is already a composite metric; we map it to S_sentiment.
    raw_ts = ld.get_history(
        universe=[smoke_ric],
        fields=TRNA_TS_FIELDS,
        interval="1W",
        start=START,
        end=END,
    )
    if isinstance(raw_ts.index, pd.MultiIndex):
        raw_ts = raw_ts.reset_index()
    elif isinstance(raw_ts.index, pd.DatetimeIndex):
        raw_ts = raw_ts.reset_index().rename(columns={"Date": "week_start", "index": "week_start"})

    # Standardise column names
    col_map = {}
    for col in raw_ts.columns:
        lower = str(col).lower()
        if "date" in lower or lower == "index":
            col_map[col] = "week_start"
        elif "buzz" in lower and "avg" not in lower:
            col_map[col] = "buzz"
        elif "sentiment" in lower:
            col_map[col] = "sentiment_score"
    raw_ts = raw_ts.rename(columns=col_map)

    if "week_start" in raw_ts.columns:
        raw_ts["week_start"] = pd.to_datetime(raw_ts["week_start"])
        raw_ts = raw_ts.sort_values("week_start")

    weekly_sentiment = raw_ts.copy()
    print(f"Weekly TRNA rows: {len(weekly_sentiment)}")
    weekly_sentiment.head()

In [ ]:

# ── Path B: per-article TRNA via headline metadata ────────────────────────────
# Iterates month by month and normalises each row to the canonical raw schema:
#   article_time, headline, story_id, source_code, article_date,
#   relevance_score, pos, neg, obj

articles_with_trna = None

if FETCH_MODE == "headline_trna":
    chunks = month_chunks(START, END)
    query  = build_news_query(smoke_ric)
    chunk_frames = []

    with tqdm(chunks, desc=f"{TICKER} headlines", unit="month") as pbar:
        for chunk_start, chunk_end in pbar:
            pbar.set_postfix_str(chunk_start.strftime("%Y-%m"))
            start_ts = chunk_start.to_pydatetime()
            end_ts   = (chunk_end + pd.Timedelta(days=1)).to_pydatetime()
            try:
                raw = ld.news.get_headlines(query, start=start_ts, end=end_ts, count=10_000)
            except Exception:
                continue
            if raw is None:
                continue
            frame = raw if isinstance(raw, pd.DataFrame) else getattr(raw, "data", raw)
            if isinstance(frame, pd.DataFrame) and not frame.empty:
                chunk_frames.append(frame)

    if not chunk_frames:
        print("No headlines returned.")
    else:
        all_raw = pd.concat(chunk_frames, ignore_index=True)

        # ── Canonical raw column mapping ──────────────────────────────────────
        col_map = {}
        for col in all_raw.columns:
            c = str(col).lower()
            if c in {"versioncreated", "date", "article_time"}:
                col_map[col] = "article_time"
            elif c in {"headline", "title"}:
                col_map[col] = "headline"
            elif c in {"storyid", "story_id"}:
                col_map[col] = "story_id"
            elif c in {"sourcecode", "source_code"}:
                col_map[col] = "source_code"
            # TRNA sentiment probability columns
            elif c in {"sentimentpositive", "positiveprob", "pos"}:
                col_map[col] = "pos"
            elif c in {"sentimentnegative", "negativeprob", "neg"}:
                col_map[col] = "neg"
            elif c in {"sentimentneutral", "neutralprob", "obj"}:
                col_map[col] = "obj"
            elif c in {"relevance", "relevancescore"}:
                col_map[col] = "relevance_score"
        all_raw = all_raw.rename(columns=col_map)

        # ── Normalise article_time ────────────────────────────────────────────
        all_raw["article_time"] = pd.to_datetime(
            all_raw.get("article_time"), utc=True, errors="coerce", format="mixed"
        )

        # ── Derive article_date (Eastern time, date only) ─────────────────────
        all_raw["article_date"] = (
            all_raw["article_time"]
            .dt.tz_convert("America/New_York")
            .dt.normalize()
            .dt.tz_localize(None)
        )

        # ── Deduplicate on story_id ────────────────────────────────────────────
        if "story_id" in all_raw.columns:
            all_raw = all_raw.drop_duplicates(subset=["story_id"])

        # ── Keep only canonical raw columns ───────────────────────────────────
        RAW_COLS = ["article_time", "headline", "story_id", "source_code",
                    "article_date", "relevance_score", "pos", "neg", "obj"]
        keep = [c for c in RAW_COLS if c in all_raw.columns]
        articles_with_trna = all_raw[keep].sort_values("article_time").reset_index(drop=True)

        print(f"Total articles: {len(articles_with_trna)}")
        print(f"Columns       : {list(articles_with_trna.columns)}")
        articles_with_trna.head(3)



## Step 3 — Add calculated fields

### Raw dataset schema

| Column | Type | Source |
|---|---|---|
| `article_time` | datetime (UTC) | LSEG headline `versionCreated` |
| `headline` | str | LSEG news headline text |
| `story_id` | str | LSEG unique story identifier |
| `source_code` | str | News source code |
| `article_date` | date | `article_time` converted to Eastern, date only |
| `relevance_score` | float [0–1] | TRNA relevance of article to the stock |
| `pos` | float [0–1] | TRNA positive sentiment probability |
| `neg` | float [0–1] | TRNA negative sentiment probability |
| `obj` | float [0–1] | TRNA neutral (objective) sentiment probability |

### Calculated fields

```
sentiment_score = relevance_score × (pos - neg)     # Eq. 8, Song et al. 2017
week_start      = Monday of the article's calendar week
```


In [ ]:

# ── Calculated fields: per-article sentiment score ────────────────────────────
# Adds sentiment_score on top of the raw schema.
# Raw  : article_time, headline, story_id, source_code, article_date,
#         relevance_score, pos, neg, obj
# Calc : sentiment_score, week_start

if FETCH_MODE == "headline_trna" and articles_with_trna is not None:
    df = articles_with_trna.copy()

    for col in ["pos", "neg", "obj", "relevance_score"]:
        df[col] = pd.to_numeric(df.get(col), errors="coerce")

    # Eq. 8 — paper formula
    df["sentiment_score"] = df["relevance_score"] * (df["pos"] - df["neg"])

    # Week bucket for later aggregation (ISO Monday-anchored)
    df["week_start"] = df["article_date"] - pd.to_timedelta(
        df["article_date"].dt.dayofweek, unit="D"
    )

    articles_with_trna = df

    print(f"Sentiment score range: [{df['sentiment_score'].min():.3f}, {df['sentiment_score'].max():.3f}]")
    df[["article_time", "headline", "story_id", "source_code",
        "article_date", "relevance_score", "pos", "neg", "obj", "sentiment_score"]].head(3)


In [ ]:
# ── Aggregate to weekly sentiment ─────────────────────────────────────────────
if FETCH_MODE == "headline_trna" and articles_with_trna is not None:
    weekly_sentiment = (
        articles_with_trna
        .groupby("week_start", as_index=False)
        .agg(
            sentiment_score=("sentiment_score", "mean"),
            buzz=("sentiment_score", "count"),
            avg_pos=("pos", "mean"),
            avg_neg=("neg", "mean"),
            avg_relevance=("relevance", "mean"),
        )
        .sort_values("week_start")
    )

    # Fill missing weeks with NaN (no-news weeks)
    all_weeks = pd.date_range(START, END, freq="W-MON").normalize()
    weekly_sentiment = (
        weekly_sentiment
        .set_index("week_start")
        .reindex(all_weeks)
        .rename_axis("week_start")
        .reset_index()
    )
    weekly_sentiment["buzz"] = weekly_sentiment["buzz"].fillna(0).astype(int)

    print(f"Weekly rows: {len(weekly_sentiment)}")
    print(f"Weeks with news: {(weekly_sentiment['buzz'] > 0).sum()}")
    weekly_sentiment.head(10)

## Step 4 — Compute Relative Sentiment Indicators (RSI)

Feature | Formula
---|---
**Shock** | `(S(t) - μ_{t-N..t-1}) / σ_{t-N..t-1}`
**Trend** | `Σ ΔS(i)` for `i ∈ [t-N, t-1]`
**1-week avg** | `S(t-1)` (previous week's sentiment)
**1-month avg** | `mean(S(t-4), …, S(t-1))`

In [ ]:
# ── Compute RSI features ───────────────────────────────────────────────────────
if weekly_sentiment is not None:
    df = weekly_sentiment.copy().sort_values("week_start").reset_index(drop=True)
    s = df["sentiment_score"]

    # Rolling window (N weeks, excluding current)
    roll_mean = s.shift(1).rolling(N_WEEKS, min_periods=max(1, N_WEEKS // 2)).mean()
    roll_std  = s.shift(1).rolling(N_WEEKS, min_periods=max(2, N_WEEKS // 2)).std(ddof=1)

    df["shock"] = (s - roll_mean) / roll_std.replace(0, np.nan)

    # Trend: sum of ΔS over the past N periods
    delta_s = s.diff()
    df["trend"] = delta_s.shift(1).rolling(N_WEEKS, min_periods=1).sum()

    # Leading sentiment averages (using lagged values so there's no look-ahead)
    df["avg_sentiment_1w"]  = s.shift(1)                                   # previous week
    df["avg_sentiment_1m"]  = s.shift(1).rolling(4, min_periods=1).mean()  # ~1 month

    rsi_features = df[["week_start", "sentiment_score", "buzz",
                        "shock", "trend", "avg_sentiment_1w", "avg_sentiment_1m"]].copy()
    rsi_features["ticker"] = TICKER
    rsi_features["ric"]    = smoke_ric
    rsi_features["n_weeks"] = N_WEEKS

    print(f"RSI features shape: {rsi_features.shape}")
    rsi_features.dropna(subset=["shock", "trend"]).head(10)

In [ ]:
# ── Quick sanity plot ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

if weekly_sentiment is not None:
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    axes[0].plot(rsi_features["week_start"], rsi_features["sentiment_score"], lw=0.8)
    axes[0].axhline(0, color="grey", lw=0.5, ls="--")
    axes[0].set_ylabel("Weekly sentiment")
    axes[0].set_title(f"{TICKER} — Sentiment Indicators (N={N_WEEKS} weeks)")

    axes[1].plot(rsi_features["week_start"], rsi_features["shock"], lw=0.8, color="tab:orange")
    axes[1].axhline(0, color="grey", lw=0.5, ls="--")
    axes[1].set_ylabel("Shock (z-score)")

    axes[2].plot(rsi_features["week_start"], rsi_features["trend"], lw=0.8, color="tab:green")
    axes[2].axhline(0, color="grey", lw=0.5, ls="--")
    axes[2].set_ylabel("Trend (Σ ΔS)")

    fig.tight_layout()
    plt.show()

## Step 5 — Save outputs

In [ ]:
# ── Save outputs ───────────────────────────────────────────────────────────────
saved = []

if FETCH_MODE == "headline_trna" and articles_with_trna is not None:
    articles_with_trna.to_parquet(ARTICLE_PARQUET, index=False)
    saved.append(str(ARTICLE_PARQUET))

if weekly_sentiment is not None:
    weekly_sentiment.to_parquet(WEEKLY_PARQUET, index=False)
    saved.append(str(WEEKLY_PARQUET))

if weekly_sentiment is not None and 'rsi_features' in dir():
    rsi_features.to_parquet(FEATURES_PARQUET, index=False)
    saved.append(str(FEATURES_PARQUET))

summary = {
    "ticker":      TICKER,
    "ric":         smoke_ric,
    "start":       START,
    "end":         END,
    "fetch_mode":  FETCH_MODE,
    "n_weeks":     N_WEEKS,
    "weekly_rows": int(len(weekly_sentiment)) if weekly_sentiment is not None else 0,
    "features_non_null": int(rsi_features[["shock", "trend"]].dropna().shape[0]) if 'rsi_features' in dir() else 0,
    "outputs":     saved,
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2))

print("Saved:")
for p in saved:
    print(f"  {p}")
print(f"  {SUMMARY_JSON}")
print()
print(json.dumps(summary, indent=2))

## Note — If TRNA is unavailable

If `FETCH_MODE == 'unavailable'`, the LSEG subscription does not include pre-computed TRNA analytics.
The paper explicitly states it used **Thomson Reuters News Analytics** (a paid product separate from
the standard Eikon/Workspace news feed).

**Next steps if TRNA access is missing:**

1. **Request TRNA access** — Ask U of T Map & Data Library or LSEG account manager for the
   "News Analytics" add-on. The package name is `TR.News.TRNA.*`.

2. **FinBERT proxy** — Run [ProsusAI/finbert](https://huggingface.co/ProsusAI/finbert) on the
   article text fetched via `notebooks/fetch_news_articles.ipynb`. FinBERT outputs
   positive/negative/neutral probabilities directly comparable to TRNA's `pos/neg/obj`.
   Relevance can be approximated as 1.0 or via a keyword-overlap heuristic.

3. **VADER proxy** — Simpler alternative; less accurate for financial text but available offline.

Option 2 (FinBERT) most closely approximates the paper's methodology when TRNA is not accessible.